# Stop-sign violation detector — Colab runner

This notebook is dedicated to Yuval's stop-sign module. It checks out the `stop-sign-module` branch, runs the synthetic tests, processes one video with only the stop-sign module enabled, renders the stop zones, and saves the alerts as JSON.

For a meaningful result, use footage where a stop sign and the approaching vehicle are both visible. A generic motorway sample can verify that the pipeline runs, but it cannot validate stop-sign behaviour.

In [36]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. In Colab choose Runtime > Change runtime type > GPU.")

CUDA available: True
Device: Tesla T4


## 1. Fetch the Stop Sign branch

This is intentionally separate from Ariel's `run_on_colab.ipynb`. Re-running the cell updates an existing clean checkout with a fast-forward pull.

In [37]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Arielevi15/Crime_Traffic_Dedector.git"
# The branch under review, not main: `ego_corridor_fraction` and the widened
# stop zone live here until PR #6 merges. Pointing this at main gives a
# TypeError from `run()` for an argument that main's pipeline has never seen.
BRANCH = "validate-stopsign-real-video"
REPO_DIR = "/content/Crime_Traffic_Dedector"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# Fetched before the checkout because a branch pushed after this clone was
# made is not among its remote refs yet, and `checkout` alone would fail.
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)

os.chdir(REPO_DIR)

# Drop the package from Python's import cache. Without this, the checkout
# updates the files on disk while the kernel keeps executing the copies
# already in memory -- a fixed bug reproduces identically and the fix
# looks like it failed.
for _name in [n for n in list(sys.modules) if n.startswith("road_crime")]:
    sys.modules.pop(_name, None)

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("Repository:", REPO_DIR)
print("Branch:", BRANCH)
print("Commit:", commit)

# Proves the checkout reached the kernel, rather than trusting that it did.
import inspect

from road_crime.pipeline import run as _run

assert "ego_corridor_fraction" in inspect.signature(_run).parameters, (
    "This checkout's pipeline.run has no ego_corridor_fraction -- the branch "
    "above did not reach the kernel. Restart the runtime and re-run this cell."
)
print("pipeline.run accepts ego_corridor_fraction: OK")

Repository: /content/Crime_Traffic_Dedector
Branch: validate-stopsign-real-video
Commit: 4b60748
pipeline.run accepts ego_corridor_fraction: OK


## 2. Install dependencies and run the synthetic tests

In [38]:
%pip install -q rfdetr trackers

In [39]:
import os
import subprocess
import sys

print("cwd:", os.getcwd())
print("python:", sys.executable)

# Diagnostic, kept because the symptom is otherwise baffling: a bare
# "No module named tests.test_stop_sign" while tests/ is plainly present.
probe = subprocess.run(
    [sys.executable, "-c", "import tests; print(tests.__file__ or list(tests.__path__))"],
    capture_output=True,
    text=True,
)
print("`import tests` resolves to:", (probe.stdout or probe.stderr).strip())

# Both suites, not just this one: main stays green only if the other
# track's tests pass too (WORKPLAN rule 0.6).
#
# Invoked by file path rather than `-m tests.…` on purpose. tests/ has no
# __init__.py, so it is a namespace package, and a *regular* package named
# `tests` anywhere on sys.path beats it even when the repository root comes
# first -- which is what the Colab image has. Each suite already puts the
# repository root on sys.path itself, so a path invocation is equivalent
# and immune to that shadowing.
#
# Output is captured and reprinted so a failure shows its reason: an
# uncaptured subprocess writes to the kernel's stdout, not to this cell.
failed = []
for suite in ("tests/test_stop_sign.py", "tests/test_wrong_way.py"):
    print("\n=== {0} ===".format(suite))
    result = subprocess.run([sys.executable, suite], capture_output=True, text=True)
    print(result.stdout, end="")
    if result.returncode != 0:
        print(result.stderr, end="")
        failed.append("{0} (exit {1})".format(suite, result.returncode))

if failed:
    raise SystemExit("Failing suites: " + ", ".join(failed))
print("\nBoth suites passed.")

cwd: /content/Crime_Traffic_Dedector
python: /usr/bin/python3
`import tests` resolves to: /usr/local/lib/python3.12/dist-packages/tests/__init__.py

=== tests/test_stop_sign.py ===
ok   test_alert_and_evidence_are_finite_strict_json
ok   test_alert_has_complete_auditable_evidence
ok   test_alert_is_emitted_only_after_vehicle_exits_zone
ok   test_corridor_is_off_by_default
ok   test_default_config_stationary_vehicle_never_alerts
ok   test_default_required_length_stop_never_uses_pre_zone_speed
ok   test_detector_config_is_a_compatibility_alias
ok   test_insufficient_history_never_becomes_infinite_speed_evidence
ok   test_late_stop_clears_earlier_fast_motion
ok   test_narrow_default_zone_misses_the_roadway_beside_the_sign
ok   test_reentering_an_evaluated_zone_does_not_duplicate_alert
ok   test_rolling_stop_above_threshold_is_flagged
ok   test_short_visit_is_locked_without_alert_or_later_rejudgment
ok   test_stop_sign_tracker_handles_jitter_reordering_and_dropout
ok   test_stop_zone_heuri

## 3. Choose a video

Run **one** of the next three input cells.

- **Option A** — upload from your computer. Colab web UI only; it hangs in VS Code.
- **Option B** — a file you have already put in Google Drive.
- **Option C** — fetch the verified stop-sign clip from YouTube. Needs no
  browser frontend, so this is the one to use from VS Code.

In [ ]:
# Option A — upload a short clip from your computer.
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No video was uploaded.")
VIDEO = os.path.abspath(next(iter(uploaded)))
print("Video:", VIDEO)

In [40]:
# Option B — a clip you have put in Google Drive. Works from VS Code.
import glob
import os

from google.colab import drive

drive.mount("/content/drive")

# Searched rather than hardcoded: a fixed path is the usual reason this cell
# fails, and the clip is easier to find than to spell. Any folder under My
# Drive will do, so it does not matter where it was dropped.
matches = sorted(glob.glob("/content/drive/MyDrive/Traffic Crimes Model Detector Project/Dashcam/dmv_segment_intersection.mp4", recursive=True))
if not matches:
    raise FileNotFoundError(
        "No .mp4 anywhere under /content/drive/MyDrive. Upload the clip to "
        "Drive first (My Drive, any folder), then re-run this cell."
    )

print("Found in Drive:")
for index, path in enumerate(matches):
    print("  [{0}] {1} ({2:.1f} MB)".format(index, path, os.path.getsize(path) / 1e6))

# Prefers the stop-sign clip by name, else falls back to the first match.
preferred = [path for path in matches if "dmv" in os.path.basename(path).lower()]
VIDEO = (preferred or matches)[0]
print("\nVideo:", VIDEO)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found in Drive:
  [0] /content/drive/MyDrive/Traffic Crimes Model Detector Project/Dashcam/dmv_segment_intersection.mp4 (9.2 MB)

Video: /content/drive/MyDrive/Traffic Crimes Model Detector Project/Dashcam/dmv_segment_intersection.mp4


### Option C — fetch from YouTube (blocked on Colab)

Kept as a record, not as a route. YouTube rejects Colab's datacenter IP ranges
with "Sign in to confirm you're not a bot", so this fails regardless of the
`yt-dlp` options used. Cookies can be passed but do not reliably survive the
IP-based block. Use Option B from a Colab runtime.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yt-dlp"], check=True)

import cv2
import yt_dlp

YOUTUBE_URL = "https://www.youtube.com/watch?v=-ps3WzTHkDY"
RAW_PATH = "/content/dmv_fail_raw.mp4"
SEGMENT_PATH = "/content/dmv_segment_intersection.mp4"
START_S, END_S = 275, 325

# Audio is irrelevant here -- OpenCV reads frames only -- so a video-only
# stream keeps this to one file and avoids the ffmpeg merge step.
options = {
    "format": "bestvideo[height<=720][ext=mp4]/best[ext=mp4]/best",
    "outtmpl": RAW_PATH,
    "quiet": True,
    "overwrites": True,
}
with yt_dlp.YoutubeDL(options) as downloader:
    downloader.download([YOUTUBE_URL])

capture = cv2.VideoCapture(RAW_PATH)
if not capture.isOpened():
    raise RuntimeError("OpenCV could not open the download: {0}".format(RAW_PATH))

fps = capture.get(cv2.CAP_PROP_FPS)
width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
capture.set(cv2.CAP_PROP_POS_FRAMES, int(START_S * fps))

writer = cv2.VideoWriter(
    SEGMENT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
)
written = 0
for _ in range(int((END_S - START_S) * fps)):
    ok, frame = capture.read()
    if not ok:
        break
    writer.write(frame)
    written += 1
capture.release()
writer.release()

if written == 0:
    raise RuntimeError(
        "No frames written -- check START_S against the length of the clip."
    )

VIDEO = SEGMENT_PATH
print("Source: {0}x{1} @ {2:.2f} fps".format(width, height, fps))
print("Trimmed {0}s-{1}s -> {2} frame(s)".format(START_S, END_S, written))
print("Video:", VIDEO)

## 4. Run only the Stop Sign module

The RF-DETR class table used by this project maps class ID `13` to `stop sign`; ID `11` is `fire hydrant` and must not be accepted as a stop sign.

In [41]:
from road_crime.pipeline import run
from road_crime.stop_sign_detector import StopSignConfig

OUTPUT = "/content/stop_sign_result.mp4"
ALERTS_JSON = "/content/stop_sign_alerts.json"
MODEL_VARIANT = "nano"
CONFIDENCE = 0.35
LIMIT_FRAMES = None  # Set an integer such as 300 for a quick smoke test.

# Both settings below come from what the first real clip showed, not from
# taste. At the default zone_width_scale of 2.0 the zone sits on the verge
# beside the sign rather than on the carriageway, so a vehicle that visibly
# crossed without stopping was never judged at all -- no decision, no alert.
stop_config = StopSignConfig(
    zone_width_scale=10.0,
    zone_min_width_px=200.0,
)

alerts = run(
    video=VIDEO,
    output=OUTPUT,
    variant=MODEL_VARIANT,
    conf=CONFIDENCE,
    limit_frames=LIMIT_FRAMES,
    modules=("stop_sign",),
    stop_sign_config=stop_config,
    # A zone wide enough to reach our carriageway also covers the crossing
    # road, whose vehicles answer to a sign whose blank back is all this
    # camera can see. Judging those would be guesswork, so they are excluded.
    ego_corridor_fraction=0.5,
)

print(f"Finished with {len(alerts)} alert(s).")

[2026-08-11 20:15:39] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-nano.pth already exists with correct MD5 hash.


[2026-08-11 20:15:39] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-08-11 20:15:39] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-08-11 20:15:41] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-nano.pth already exists with correct MD5 hash.


[2026-08-11 20:15:42] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.inference(dtype=torch.float16).


Ego corridor: x in [160, 480] of 640px wide frame -- vehicles outside it are not judged.

--- class ids seen in the first 30 frame(s) ---
  names come from the model itself

            id     name                   detections
  [TRACKED] 3      car                    41
  [       ] 13     stop sign              4
  [       ] 1      person                 1

  VEHICLE_CLASS_IDS is [3, 4, 6, 8] (car, motorcycle, bus, truck under the
  91-class scheme). If the names beside those ids are not those
  vehicles, fix the constant before trusting a single alert.

  frame 100, 0 alert(s) so far
  frame 200, 0 alert(s) so far
  frame 300, 0 alert(s) so far
  frame 400, 0 alert(s) so far
  frame 500, 0 alert(s) so far
  frame 600, 0 alert(s) so far
  frame 700, 0 alert(s) so far
  frame 800, 0 alert(s) so far
  frame 900, 0 alert(s) so far
  frame 1000, 0 alert(s) so far
  frame 1100, 0 alert(s) so far
  frame 1200, 0 alert(s) so far
  frame 1300, 0 alert(s) so far
  frame 1400, 0 alert(s) so far

In [42]:
import json
from pprint import pprint

with open(ALERTS_JSON, "w", encoding="utf-8") as handle:
    json.dump(alerts, handle, indent=2, ensure_ascii=False, allow_nan=False)

pprint(alerts)
print("Alerts JSON:", ALERTS_JSON)

[]
Alerts JSON: /content/stop_sign_alerts.json


## 5. Preview and download the result

In [43]:
from IPython.display import Video, display

H264_OUTPUT = "/content/stop_sign_result_h264.mp4"
subprocess.run(
    ["ffmpeg", "-loglevel", "error", "-i", OUTPUT, "-vcodec", "libx264", "-y", H264_OUTPUT],
    check=True,
)
display(Video(H264_OUTPUT, embed=True, width=900))

In [ ]:
from google.colab import files

files.download(H264_OUTPUT)
files.download(ALERTS_JSON)

In [ ]:
# Copy the results to Drive rather than downloading them. Use this from VS
# Code: `files.download` above is another `eval_js` call, so it needs the
# Colab web UI and does nothing here. Download from Drive in a browser after.
import os
import shutil

from google.colab import drive

drive.mount("/content/drive")

DEST_DIR = "/content/drive/MyDrive/Traffic Crimes Model Detector Project/Dashcam"
os.makedirs(DEST_DIR, exist_ok=True)

for source in (H264_OUTPUT, ALERTS_JSON):
    if not os.path.isfile(source):
        raise FileNotFoundError(
            "{0} is missing -- run section 4 and the ffmpeg cell first.".format(source)
        )
    destination = os.path.join(DEST_DIR, os.path.basename(source))
    shutil.copyfile(source, destination)
    print(
        "{0} -> {1} ({2:.1f} MB)".format(
            os.path.basename(source), destination, os.path.getsize(destination) / 1e6
        )
    )

print("
Now open drive.google.com and download them from that folder.")

## Reading the output

- Yellow rectangles are the inferred stop zones.
- A red `STOP SIGN` label is drawn only after a tracked vehicle exits a zone without a measured full stop.
- Zero alerts is not automatically success: confirm that RF-DETR detected the sign, that the zone overlaps the vehicle path, and that the complete approach/exit is present in the clip.
- Threshold tuning and real-footage acceptance are still pending; keep this PR in Draft until those checks are complete.